In [3]:
# Install if needed: pip install qdrant-client openai
# IMPORTANT: CAMEL-AI's QdrantStorage expects a default (unnamed) vector, not a named vector
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os

load_dotenv()

client = QdrantClient(
    os.getenv("QDRANT_URL"),
    # api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True,
    timeout=30.0
)

In [2]:
# Create collection with default (unnamed) vector for CAMEL-AI compatibility
# Note: If you need to fix an existing collection, use: make fix-qdrant-collection
client.create_collection(
    collection_name="log_fixes_2",
    vectors_config=models.VectorParams(
        size=1536,  # text-embedding-3-small dimension
        distance=models.Distance.COSINE
    )
    # Note: Sparse vectors (BM25) can be added later if needed, but CAMEL-AI uses dense vectors
)

True

In [8]:
# Initialize OpenAI embedding model: text-embedding-3-small
from camel.embeddings import OpenAIEmbedding
from camel.types import EmbeddingModelType

# Get OpenAI API credentials (can use separate embedding credentials or fall back to main OpenAI config)
openai_base_url = os.getenv("OPENAI_EMBEDDING_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"))
openai_api_key = os.getenv("OPENAI_EMBEDDING_API_KEY", os.getenv("OPENAI_API_KEY"))

# Initialize OpenAI embedding model: text-embedding-3-small (1536 dimensions)
embedding = OpenAIEmbedding(
    model_type=EmbeddingModelType.TEXT_EMBEDDING_3_SMALL,
    url=openai_base_url,
    api_key=openai_api_key
)

print("OpenAI embedding model initialized: text-embedding-3-small")


OpenAI embedding model initialized: text-embedding-3-small


In [9]:
# Example query strings for testing RAG search functionality
EXAMPLE_QUERIES = [
    "Connection timeout when calling external API",
    "Database connection failed",
    "Bank gateway timeout after 5000ms",
    "Database connection pool exhausted",
    "Failed to authenticate user: Invalid API key",
    "Connection refused: Unable to connect to Redis server",
    "Out of memory: Java heap space",
    "Service unavailable error",
    "Network timeout error",
    "Authentication failed",
    "Database query timeout",
    "Connection pool exhausted",
    "Memory leak detected",
    "API rate limit exceeded",
    "SSL certificate verification failed"
]

print(f"Example queries loaded: {len(EXAMPLE_QUERIES)} queries")
for i, query in enumerate(EXAMPLE_QUERIES[:5], 1):
    print(f"{i}. {query}")


Example queries loaded: 15 queries
1. Connection timeout when calling external API
2. Database connection failed
3. Bank gateway timeout after 5000ms
4. Database connection pool exhausted
5. Failed to authenticate user: Invalid API key


In [ ]:
# Example: Generate embedding for a query and search the collection
# Note: Using query_points() instead of deprecated search() method
query_text = EXAMPLE_QUERIES[0]
print(f"Query: {query_text}")

# Generate embedding vector using OpenAI text-embedding-3-small
query_vector = embedding.embed(query_text)
print(f"Embedding dimension: {len(query_vector)}")

# Search the collection using modern query_points API
results = client.query_points(
    collection_name="log_fixes_2",
    query=query_vector,
    limit=5,  # Return top 5 results
    with_payload=True  # Include payload in results
)

print(f"\nFound {len(results.points)} results:")
for i, point in enumerate(results.points, 1):
    print(f"\n{i}. Score: {point.score:.4f}")
    print(f"   ID: {point.id}")
    if point.payload:
        # Extract fix if available, otherwise show full payload
        fix = point.payload.get('fix', point.payload.get('text', 'N/A'))
        error = point.payload.get('error', 'N/A')
        print(f"   Error: {error[:80]}...")
        print(f"   Fix: {fix[:100]}..." if len(str(fix)) > 100 else f"   Fix: {fix}")


Query: Database error, connection problem


NameError: name 'embedding' is not defined

In [6]:
# Test multiple example queries using modern API
for query in EXAMPLE_QUERIES[:3]:  # Test first 3 queries
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    # Generate embedding
    query_vector = embedding.embed(query)
    
    # Search using query_points (modern API)
    results = client.query_points(
        collection_name="log_fixes_2",
        query=query_vector,
        limit=3,
        with_payload=True
    )
    
    print(f"Top {len(results.points)} results:")
    for i, point in enumerate(results.points, 1):
        fix_preview = ""
        if point.payload:
            fix = point.payload.get('fix', point.payload.get('text', ''))
            fix_preview = f" | Fix: {fix[:50]}..." if fix else ""
        print(f"  {i}. Score: {point.score:.4f} | ID: {point.id}{fix_preview}")


NameError: name 'EXAMPLE_QUERIES' is not defined